In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
from scipy.io import loadmat
import netCDF4 as nc
from skimage.transform import resize
import os
from scipy.ndimage import gaussian_filter, median_filter, uniform_filter
import gc
from scipy.ndimage import median_filter
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


# 设置参数
n_blocks = 3
n_layers_per_block = 5
n_neurons = 100
batch_size = 512
dropout_rate = 0.3  # Dropout比率
l2_lambda = 0.01  # L2 正则化因子

# 构建特征提取模块时加上L2正则项和Dropout
def feature_block(input_dim, regularizer=None, dropout_rate=0.3):
    model = tf.keras.Sequential()
    for _ in range(n_layers_per_block):
        model.add(tf.keras.layers.Dense(
            n_neurons, activation='tanh', kernel_regularizer=regularizer))
        model.add(tf.keras.layers.Dropout(dropout_rate))  # 加入 Dropout 层
    return model


# PINN模型结构
class PINN(tf.keras.Model):
    def __init__(self, input_dim, regularizer=None, dropout_rate=0.3):
        super(PINN, self).__init__()
        # 定义三个特征提取块
        self.block1 = feature_block(
            input_dim, regularizer=regularizer, dropout_rate=dropout_rate)
        self.block2 = feature_block(
            input_dim + n_neurons, regularizer=regularizer, dropout_rate=dropout_rate)
        self.block3 = feature_block(
            input_dim + n_neurons * 2, regularizer=regularizer, dropout_rate=dropout_rate)
        self.final_layer = tf.keras.layers.Dense(2)  # 预测 u, v

    def call(self, inputs):
        # 第一个块的输出
        out1 = self.block1(inputs)

        # 第二个块的输入为第一个块的输出与原始输入拼接
        out2_input = tf.concat([inputs, out1], axis=-1)
        out2 = self.block2(out2_input)

        # 第三个块的输入为第二个块的输出、第一个块的输出与原始输入拼接
        out3_input = tf.concat([inputs, out1, out2], axis=-1)
        out3 = self.block3(out3_input)

        # 最后的输出层
        return self.final_layer(out3)
    
    
# 安全替换非数字的部分为0
def safe_replace(tensor):
    # 检查 tensor 是否有 nan 或 inf，并将其替换为 0
    return tf.where(tf.math.is_nan(tensor) | tf.math.is_inf(tensor), tf.zeros_like(tensor), tensor)


# 物理损失函数（根据网格大小进行计算）
def physics_loss(u, v, f, eta, uwind, vwind, dx, dy, g=9.81, wesize=(720, 1440)):

    u = tf.reshape(u, wesize)
    v = tf.reshape(v, wesize)
    eta = tf.reshape(eta, wesize)
    uwind = tf.reshape(uwind, wesize)
    vwind = tf.reshape(vwind, wesize)
    f = tf.reshape(f, wesize)

    dx = tf.reshape(dx, wesize)
    dy = tf.reshape(dy, wesize)

    def ddx(field):
        return (tf.roll(field, shift=-1, axis=1) - tf.roll(field, shift=1, axis=1)) / (2 * dx)

    def ddy(field):
        return (tf.roll(field, shift=-1, axis=0) - tf.roll(field, shift=1, axis=0)) / (2 * dy)

    du_dx = ddx(u)
    du_dy = ddy(u)
    dv_dx = ddx(v)
    dv_dy = ddy(v)
    deta_dx = ddx(eta)
    deta_dy = ddy(eta)


    res_u = u * du_dx + v * du_dy - f * v + g * deta_dx - f * vwind
    res_v = u * dv_dx + v * dv_dy + f * u + g * deta_dy + f * uwind
    res_u = safe_replace(res_u)
    res_v = safe_replace(res_v)

    loss_u = tf.reduce_mean(tf.square(res_u))
    loss_v = tf.reduce_mean(tf.square(res_v))
    return loss_u + loss_v





def calculate_second_derivative(h, dx, dy):
    # h 是二维数组，dx 和 dy 分别是沿 x 和 y 方向的步长（二维数组）
    ny, nx = h.shape
    
    # 计算沿 y 方向的二阶导数
    d2h_dy2 = np.zeros((ny, nx))
    # 内部点使用中差
    d2h_dy2[1:-1, 1:-1] = (h[2:, 1:-1] - 2 * h[1:-1, 1:-1] + h[:-2, 1:-1]) / (dy[1:-1, 1:-1] ** 2)
    # 边界点使用前向差分或后向差分
    d2h_dy2[0, :] = (h[1, :] - 2 * h[0, :] + h[0, :]) / (dy[0, :] ** 2)
    d2h_dy2[-1, :] = (h[-1, :] - 2 * h[-1, :] + h[-2, :]) / (dy[-1, :] ** 2)
    
    # 计算沿 x 方向的二阶导数
    d2h_dx2 = np.zeros((ny, nx))
    # 内部点使用中差
    d2h_dx2[1:-1, 1:-1] = (h[1:-1, 2:] - 2 * h[1:-1, 1:-1] + h[1:-1, :-2]) / (dx[1:-1, 1:-1] ** 2)
    # 边界点使用前向差分或后向差分
    d2h_dx2[:, 0] = (h[:, 1] - 2 * h[:, 0] + h[:, 0]) / (dx[:, 0] ** 2)
    d2h_dx2[:, -1] = (h[:, -1] - 2 * h[:, -1] + h[:, -2]) / (dx[:, -1] ** 2)
    
    return d2h_dy2, d2h_dx2

def semigeo(lats, h, x, y, dx, dy):
    # 确保 lats、x、y、dx 和 dy 是二维数组，且形状与 h 一致
    assert lats.shape == h.shape, "lats must be a 2D array with the same shape as h"
    assert x.shape == h.shape, "x must be a 2D array with the same shape as h"
    assert y.shape == h.shape, "y must be a 2D array with the same shape as h"
    assert dx.shape == h.shape, "dx must be a 2D array with the same shape as h"
    assert dy.shape == h.shape, "dy must be a 2D array with the same shape as h"
    g = 9.8
    
    # 计算 beta
    beta = 2.3*1e-11#2 * np.pi / (86400 * np.cos(np.deg2rad(lats)))
    
    # 计算二阶偏导数 ∂²h/∂y² 和 ∂²h/∂x²
    d2h_dy2, d2h_dx2 = calculate_second_derivative(h, dx, dy)
    
    # 计算 u_sg 和 v_sg
    u_sg = -g / beta * d2h_dy2
    v_sg = g / beta * d2h_dx2
    
    return u_sg, v_sg




d:\LenovoSoftstore\anaconda3\envs\py39_tensorflow\lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.12.2 when it was built against 1.12.1, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [ ]:
def generate_data(yitapath, windpath,num_lon_points=180, num_lat_points=90):

    # 生成网格坐标（经度和纬度）
    lon = np.linspace(-180, 179.75, 1440, dtype=np.float32)
    lat = np.linspace(-90, 89.75, 720, dtype=np.float32)
    x, y = np.meshgrid(lon, lat)

    # data = loadmat('ssh_1993-2016.mat')
    dx = loadmat(r'data\x.mat')
    x = dx['x']
    dy = loadmat(r'data\y.mat')
    y = dy['y']
    
    geox = loadmat(r'data\geox.mat')
    geox = geox['geox']
    geoy = loadmat(r'data\geoy.mat')
    geoy = geoy['geoy']
  
  
    f_matrix = loadmat(r'data\f_matrix.mat')
    f = f_matrix['f_matrix']
    f = np.squeeze(f)

    f = np.flipud(f)
  
    dataset = nc.Dataset(yitapath)
    # 查看文件的基本信息
    # print(dataset)
    u_geo = dataset.variables['ugos'][:][0]
    v_geo= dataset.variables['vgos'][:][0]
    sla = dataset.variables['sla'][:][0]
    land_mask = np.isnan(sla)

    sla = processdata(sla)
    u_geo = processdata(u_geo)
    v_geo = processdata(v_geo)

    dataset = nc.Dataset(windpath)
    # print(dataset)
    u_ekman = dataset.variables['ue'][:][0][0, :, :]
    v_ekman = dataset.variables['ve'][:][0][0, :, :]
    u_ekman = processdata(u_ekman)
    v_ekman = processdata(v_ekman)
  
    g = 9.8
    
    usg,vsg = semigeo(y, sla, geox, geoy, x, y)
    usg = processdata(usg)
    vsg = processdata(vsg)
    # 替换 u_geo 和 v_geo 的第 353 行到第 368 行
    # u_geo[352:368, :] = usg[352:368, :]
    # v_geo[352:368, :] = vsg[352:368, :]
    a = usg[352:368, :]
    b = vsg[352:368, :]
    
    a,b=apply_filterwith_nan(a, b, filter_type='mean', sigma=1.0, size=7)

    u_geo[352:368, :] = a
    v_geo[352:368, :] = b

    u_geo = processdata(u_geo)
    v_geo = processdata(v_geo)


    u_geo[(np.abs(u_geo) > 3) | ~np.isfinite(u_geo)] = 0
    v_geo[(np.abs(v_geo) > 3) | ~np.isfinite(v_geo)] = 0

    #Pseudo-labels
    u_true = u_geo + u_ekman
    v_true = v_geo + v_ekman

    # Resize 所有数据
    dx = resize_to_target(dx)
    dy = resize_to_target(dy)
    f = resize_to_target(f)
    u_ekman = resize_to_target(u_ekman)
    v_ekman = resize_to_target(v_ekman)
    sla = resize_to_target(sla)
    u_true = resize_to_target(u_true)
    v_true = resize_to_target(v_true)
    u_geo = resize_to_target(u_geo)
    v_geo = resize_to_target(v_geo)


    return x, y, f, u_ekman, v_ekman, sla, u_true, v_true, u_geo, v_geo

def replace_non_numeric_with_zero(arr):
    """
    将数组中的非数值元素（NaN 和 inf）替换成0

    参数：
    arr (numpy.ndarray): 输入数组

    返回：
    numpy.ndarray: 替换非数值元素后的数组
    """
    # 将非数值元素替换成0
    arr = np.where(np.isnan(arr), 0, arr)
    arr = np.where(np.isinf(arr), 0, arr)
    return arr

def apply_filterwith_nan(u, v, filter_type='gaussian', sigma=1.0, size=3):
    """
    对两个二维数组应用指定的滤波器，并保持NaN值的位置不变

    参数：
    u (numpy.ndarray): 第一个二维数组
    v (numpy.ndarray): 第二个二维数组
    filter_type (str): 滤波器类型，可选 'gaussian', 'median', 'mean'
    sigma (float): 高斯滤波器的标准差（仅适用于高斯滤波）
    size (int): 中值或平均滤波器的大小（仅适用于中值或平均滤波）

    返回：
    tuple: 滤波后的两个二维数组（ufiltered, vfiltered）
    """
    # 记录原始数组中NaN的位置
    nan_mask_u = np.isnan(u)
    nan_mask_v = np.isnan(v)
    # plt.figure()
    # plt.imshow(u)
    u = replace_non_numeric_with_zero(u)
    v = replace_non_numeric_with_zero(v)
    # plt.figure()
    # plt.imshow(u)
    # plt.show()


    if filter_type == 'gaussian':
        # 高斯滤波
        ufiltered = gaussian_filter(u, sigma=sigma)
        vfiltered = gaussian_filter(v, sigma=sigma)
    elif filter_type == 'median':
        # 中值滤波
        ufiltered = median_filter(u, size=size)
        vfiltered = median_filter(v, size=size)
    elif filter_type == 'mean':
        # 平均滤波
        ufiltered = uniform_filter(u, size=size)
        vfiltered = uniform_filter(v, size=size)
    else:
        raise ValueError("filter_type 必须为 'gaussian', 'median' 或 'mean'")

    # 将NaN的位置还原
    ufiltered[nan_mask_u] = np.nan
    vfiltered[nan_mask_v] = np.nan

    return ufiltered, vfiltered

def processdata(sla1):
    if isinstance(sla1, np.ma.MaskedArray):
        sla1 = sla1.filled()  # 将掩膜值填充为默认值（通常是填充 NaN）

    # 将字符串 '--' 转换为 NaN
    sla1 = np.where(sla1 == '--', np.nan, sla1).astype(float)

    # 将 NaN 替换为 0
    sla1 = np.nan_to_num(sla1, nan=0.0)
    # 创建掩码：绝对值大于3 或 nan 或 inf 的元素设为 0
    sla1[(np.abs(sla1) > 2) | ~np.isfinite(sla1)] = 0
    return sla1

 # 目标尺寸
target_shape = (720, 1440)


# def resize_to_target(arr, shape=target_shape):
#     return resize(arr, shape, preserve_range=True, anti_aliasing=True)


def resize_to_target(arr, shape=target_shape, dtype=np.float32):
    resized_arr = resize(arr, shape, preserve_range=True, anti_aliasing=True)
    return resized_arr.astype(dtype)


## 模型训练

In [ ]:
def train_pinn_model(model, dataset, X_data, f, sla, uekman, vekman, x, y,
                     loss_type='hybrid', learning_rate=1e-3, epochs=5000):
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # 转换为张量
    X_grid = tf.convert_to_tensor(X_data)
    f_tf = tf.convert_to_tensor(f.astype(np.float32))
    sla_tf = tf.convert_to_tensor(sla.astype(np.float32))
    uekman_tf = tf.convert_to_tensor(uekman.astype(np.float32))
    vekman_tf = tf.convert_to_tensor(vekman.astype(np.float32))
    x_tf = tf.convert_to_tensor(x)
    y_tf = tf.convert_to_tensor(y)

    data_losses = []
    phys_losses = []

    for epoch in range(epochs):
        total_loss = 0.0
        total_phys_loss = 0.0
        total_batches = 0

        for X_batch, y_batch in dataset:
            with tf.GradientTape() as tape:
                y_pred = model(X_batch)
                loss_data = tf.reduce_mean(tf.square(y_pred - y_batch))

                y_pred_grid = model(X_grid)
                utotal, vtotal = tf.split(y_pred_grid, 2, axis=-1)
                utotal = tf.reshape(utotal, [-1])
                vtotal = tf.reshape(vtotal, [-1])

                loss_phys = physics_loss(
                    utotal, vtotal, f_tf, sla_tf, uekman_tf, vekman_tf, x_tf, y_tf)

                if loss_type == 'data':
                    loss = loss_data
                elif loss_type == 'hybrid':
                    
                    ## 自适应损失，保证数值比例 1：1
       
                    adaptive_weight = tf.stop_gradient(loss_data / loss_phys)
                    
                    loss = loss_data + loss_phys * \
                           adaptive_weight
                elif loss_type == 'physics':
                    loss = loss_phys

                total_loss += loss_data.numpy()
                total_phys_loss += loss_phys.numpy()
                total_batches += 1

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))

        data_losses.append(total_loss / total_batches)
        phys_losses.append(total_phys_loss / total_batches)

        # print(
        #     f"Epoch: {epoch+1}, Data Loss: {total_loss / total_batches:.12f}, Phys Loss: {total_phys_loss / total_batches:.12f}")

    return data_losses, phys_losses




In [ ]:
import os
import scipy.io as sio
import numpy as np
import tensorflow as tf
from tensorflow.keras.regularizers import l2
import re
from tqdm import tqdm
import xarray as xr

def extract_date(filename):
    # 尝试第一种格式：带 T 的，如 20220101T1200Z
    match1 = re.search(r'(\d{8})T\d{4}Z', filename)
    if match1:
        return match1.group(1)

    # 尝试第二种格式：两个日期，如 20220101_20220701
    match2 = re.search(r'(\d{8})_(\d{8})', filename)
    if match2:
        return match2.group(1)

    # 尝试第三种通用格式：任何连续的 8 位数字
    match3 = re.search(r'(\d{8})', filename)
    if match3:
        return match3.group(1)

    


def mask_with_sla_nan(u_pred):

    sla_path = r"data\sla\2022\01\dt_global_allsat_phy_l4_20220101_20220701.nc"

    ds = xr.open_dataset(sla_path)

    sla_data = np.squeeze(ds.variables['sla'][:][0])

    ds.close()

    if sla_data.shape != u_pred.shape:
        print(f"[尺寸不匹配] SLA 和变量 shape 不一致: {sla_data.shape} vs {u_pred.shape}")
        return u_pred

    nan_mask = np.isnan(sla_data)
    u_pred[nan_mask] = np.nan

    return u_pred  

def batch_process(yita_dir, wind_dir, output_dir, generate_data, PINN, train_pinn_model,
                  l2_lambda=1e-5, dropout_rate=0.2, batch_size=128):
    yita_files = sorted([f for f in os.listdir(yita_dir) if f.endswith('.nc')])
    wind_files = sorted([f for f in os.listdir(wind_dir) if f.endswith('.nc')])
    # print(wind_files)

    for yita_file in tqdm(yita_files, ncols=150, colour='green'):
        # print(yita_file)
        yita_date = extract_date(yita_file)
      
        matched_wind_file = None


        for f in wind_files:
            wind_date = extract_date(f)
            if wind_date == yita_date:
                 matched_wind_file = f
                 break

        if not matched_wind_file:
            print(f"跳过: 找不到与 {yita_file} 匹配的风场数据")
            continue

        yitapath = os.path.join(yita_dir, yita_file)
        windpath = os.path.join(wind_dir, matched_wind_file)

     
        x, y, f, uekman, vekman, sla, uture, vture, ugeo, vgeo = generate_data(
            yitapath, windpath)
        size = 720*1440

        X_data = np.stack([ugeo, vgeo, uekman, vekman, f, sla],
                          axis=-1).astype(np.float32)
        y_data = np.stack([uture, vture], axis=-1).astype(np.float32)

        dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))
        dataset = dataset.shuffle(buffer_size=size).batch(
            batch_size).prefetch(tf.data.AUTOTUNE)

        model = PINN(
            input_dim=X_data.shape[-1], regularizer=l2(l2_lambda), dropout_rate=dropout_rate)
        model(X_data)

        # 训练
        train_pinn_model(model, dataset, X_data, f, sla,
                         uekman, vekman, x, y, 'data', 1e-3, 1000)
        train_pinn_model(model, dataset, X_data, f, sla,
                         uekman, vekman, x, y, 'hybrid', 1e-4, 1000)
        train_pinn_model(model, dataset, X_data, f, sla, uekman,
                         vekman, x, y, 'physics', 1e-5, 10000)
        # train_pinn_model(model, dataset, X_data, f, sla, uwind,
        #                  vwind, dx, dy, 'physics', 1e-6, 100)

        # model.load_weights('pinn_model_weights_final.h5')
        y_pred_test = model(X_data)
        utotal, vtotal = tf.split(y_pred_test, 2, axis=-1)
        u_pred = utotal.numpy().reshape(720, 1440)
        v_pred = vtotal.numpy().reshape(720, 1440)


        def resize_variable(name, var):
            var_tensor = tf.convert_to_tensor(
                var[..., tf.newaxis], dtype=tf.float32)
            resized = tf.image.resize(
                var_tensor, [720, 1440], method='bilinear')
            resizenp = tf.squeeze(resized).numpy()
            resizenp = mask_with_sla_nan(resizenp)
            return resizenp
        
        mat_data = {
            'ugeo': resize_variable('ugeo', ugeo),
            'vgeo': resize_variable('vgeo', vgeo),
            'uekman': resize_variable('uekman', uekman),
            'vekman': resize_variable('vekman', vekman),
            'sla': resize_variable('sla', sla),
            'utotal': resize_variable('utotal', utotal),
            'vtotal': resize_variable('vtotal', vtotal),
        }

        output_filename = f"{yita_date}.mat"
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, output_filename)
        sio.savemat(output_path, mat_data)

        tf.keras.backend.clear_session() 
        gc.collect()  


In [ ]:

import os

base_dir1 = r'data'
base_dir2 = r"data"
sla_base = os.path.join(base_dir2, 'sla')
wind_base = os.path.join(base_dir2, 'wind')
pred_base = os.path.join(base_dir1, 'pred')

In [ ]:
for year in range(2022, 2023):
    for month in range(1, 13):
        # 格式化月为两位数
        month_str = str(month).zfill(2)

        # 构建年/月路径
        sla_path = os.path.join(sla_base, str(year), month_str)
        wind_path = os.path.join(wind_base, str(year), month_str)
        print(sla_path)

        # 输出路径
        output_path = os.path.join(pred_base, str(year), month_str)
        os.makedirs(output_path, exist_ok=True)

        # 如果sla和wind路径都存在才处理
        if os.path.exists(sla_path) and os.path.exists(wind_path):
            print(f"正在处理 {year} 年 {month_str} 月的数据...")
            batch_process(sla_path, wind_path, output_path,
                          generate_data, PINN, train_pinn_model)
        else:
            print(f"跳过 {year} 年 {month_str} 月，因路径不存在。")


data\sla\2022\01
正在处理 2022 年 01 月的数据...


  0%|                                                                                                                          | 0/31 [00:00<?, ?it/s]C:\Users\28422\AppData\Local\Temp\ipykernel_25184\361021245.py:180: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  sla1 = np.where(sla1 == '--', np.nan, sla1).astype(float)


  3%|███▍                                                                                                       | 1/31 [1:31:13<45:36:37, 5473.25s/it]C:\Users\28422\AppData\Local\Temp\ipykernel_25184\361021245.py:180: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  sla1 = np.where(sla1 == '--', np.nan, sla1).astype(float)
  6%|██████▉                                                                                                    | 2/31 [3:02:44<44:10:29, 5483.76s/it]C:\Users\28422\AppData\Local\Temp\ipykernel_25184\361021245.py:180: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  sla1 = np.where(sla1 == '--', np.nan, sla1).astype(float)
 10%|██████████▎                                                                                                | 3/31 [4:34:18<42:41:14, 5488.36s/it]C:\Users\28422\AppData\Local\Temp\ipykerne